# FT-Transformer + PLR embeddings - Single Model (LB 0.95059)

A **single FT-Transformer** (Feature Tokenizer + Transformer, Gorishniy et al. 2021), in pure PyTorch,
upgraded with **PLR numerical embeddings** (Periodic-Linear-ReLU, Gorishniy et al. 2022, *On Embeddings
for Numerical Features*) to push a lone model past 0.95 Balanced Accuracy on S6E7.

**The upgrade vs a vanilla FT-Transformer:** the linear numerical tokenizer (`x*W+b`) is replaced by

$$x \;\rightarrow\; \text{ReLU}\Big(\text{Linear}\big[\sin(2\pi v x),\ \cos(2\pi v x)\big]\Big)$$

with learnable frequencies `v` per feature. This gives the transformer a much richer, multi-scale
representation of the 7 raw numerics + 39 target-encoding features -- and it measurably beats the vanilla
FT-Transformer here.

Integrated techniques:
1. **Target Encoding by exact value** across the 13 columns via `TargetEncoder(cv=5)`, **leak-free** (refit per fold) -> 39 features.
2. Transformer inputs: **7 raw numeric + 39 TE** as numeric tokens (with **PLR embeddings**) + **6 categorical** as embeddings.
3. **Class-weighted cross-entropy** (balanced) = the anti-imbalance lever.
4. **QuantileTransformer(normal)** on the numerics, fit per fold.
5. **AMP (mixed precision)** + data on GPU + **TF32**.
6. **Early stopping on Balanced Accuracy** (not loss) + best-model.

CV: StratifiedKFold(5, seed=42). Single seed. ~30 min on a single modern GPU.


In [ ]:
# ============================================================
# 1. Imports + configuration
# ============================================================
import warnings, time, json, gc, math, copy, sys
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.nn.functional as F

CFG = dict(
    n_folds=5, seed=42,
    # --- architecture ---
    d_token=128, n_blocks=2, n_heads=8,
    attn_dropout=0.2, ffn_dropout=0.1, res_dropout=0.0,
    # --- PLR numerical embeddings ---
    plr_k=24,          # number of frequencies (2k periodic features per variable)
    plr_sigma=0.1,     # std of the frequency init (the sensitive PLR hyperparameter)
    # --- optimisation ---
    batch_size=4096, lr=1e-3, weight_decay=1e-5,
    max_epochs=16, patience=4, warmup_frac=0.10, use_amp=True,
    seed_avg=[42],     # use [42, 2025] for ~+0.0003 (doubles the runtime)
    weight_search_iters=40_000,
)
SEED=CFG["seed"]; np.random.seed(SEED); torch.manual_seed(SEED)
DATA_DIR=Path("/kaggle/input/competitions/playground-series-s6e7")
OUT=Path("/kaggle/working/artifacts"); OUT.mkdir(exist_ok=True, parents=True)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
if DEVICE=="cuda":
    torch.backends.cudnn.benchmark=True
    torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -> CPU training (very slow).")
print("device:", DEVICE); print(json.dumps(CFG, indent=0))


In [ ]:
# ============================================================
# 2. Data + encoding + folds
# ============================================================
train=pd.read_csv(DATA_DIR/"train.csv"); test=pd.read_csv(DATA_DIR/"test.csv")
ID,TARGET="id","health_condition"
CLASSES=["at-risk","fit","unhealthy"]; C2I={c:i for i,c in enumerate(CLASSES)}
I2C={i:c for c,i in C2I.items()}; N_CLASSES=3
y=train[TARGET].map(C2I).to_numpy()
NUM=["sleep_duration","heart_rate","bmi","calorie_expenditure","step_count","exercise_duration","water_intake"]
CAT=["diet_type","stress_level","sleep_quality","physical_activity_level","smoking_alcohol","gender"]
ALLC=NUM+CAT
from sklearn.model_selection import StratifiedKFold
skf=StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=SEED)
FOLDS=list(skf.split(train,y)); priors=np.bincount(y)/len(y)
print("train:",train.shape,"| test:",test.shape,"| priors:",dict(zip(CLASSES,priors.round(4))))
# categorical codes (union train+test, NaN = own category)
cat_maps,CAT_CARD={},[]
for c in CAT:
    vals=sorted(set(train[c].fillna("__nan__").astype(str))|set(test[c].fillna("__nan__").astype(str)))
    cat_maps[c]={v:i for i,v in enumerate(vals)}; CAT_CARD.append(len(vals))
def cat_codes(df):
    a=np.zeros((len(df),len(CAT)),np.int64)
    for j,c in enumerate(CAT):
        a[:,j]=df[c].fillna("__nan__").astype(str).map(cat_maps[c]).fillna(0).astype(np.int64).values
    return a
XC_full=cat_codes(train); XC_test=cat_codes(test)
# exact-value string versions of the 13 columns for target encoding
Xs_full=pd.DataFrame({c:train[c].fillna("__nan__").astype(str) for c in ALLC})
Xs_test=pd.DataFrame({c:test[c].fillna("__nan__").astype(str)  for c in ALLC})
te_names=[f"te_{c}_{k}" for c in ALLC for k in range(N_CLASSES)]
print("num tokens:",len(NUM)+len(te_names),"| cat tokens:",len(CAT),"| cat cardinalities:",CAT_CARD)


In [ ]:
# ============================================================
# 3. Metric + bias monitoring + decision rule
# ============================================================
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
def fast_balacc(y_true,y_pred,n=N_CLASSES):
    r=np.empty(n)
    for c in range(n):
        m=(y_true==c); r[c]=(y_pred[m]==c).mean() if m.any() else np.nan
    return np.nanmean(r)
def per_class_recall(y_true,y_pred):
    return {CLASSES[c]:round(float((y_pred[y_true==c]==c).mean()),4) for c in range(N_CLASSES)}
def bias_report(y_true,y_pred,title=""):
    print(f"\n--- BIAS MONITOR {title} ---")
    print("balanced acc :",round(fast_balacc(y_true,y_pred),5))
    print("recall/class :",per_class_recall(y_true,y_pred))
    cm=confusion_matrix(y_true,y_pred); cmn=cm/cm.sum(1,keepdims=True)
    print("confusion (row-norm, rows=true):")
    print("            "+"  ".join(f"{c:>9s}" for c in CLASSES))
    for i,c in enumerate(CLASSES):
        print(f"  {c:>9s} "+"  ".join(f"{cmn[i,j]:9.4f}" for j in range(N_CLASSES)))
def optimize_decision_weights(proba,y_true,n_iter=CFG["weight_search_iters"],seed=SEED,verbose=False):
    proba=proba.astype(np.float32); idx=[np.where(y_true==c)[0] for c in range(N_CLASSES)]
    def ba(w):
        p=(proba*w).argmax(1)
        return ((p[idx[0]]==0).mean()+(p[idx[1]]==1).mean()+(p[idx[2]]==2).mean())/3
    rng=np.random.default_rng(seed); bw=np.ones(N_CLASSES,np.float32); best=ba(bw)
    w0=(1/priors).astype(np.float32)
    if ba(w0)>best: best,bw=ba(w0),w0
    for _ in range(n_iter):
        w=rng.uniform(0.15,6.0,N_CLASSES).astype(np.float32); s=ba(w)
        if s>best: best,bw=s,w
    for _ in range(6):
        for c in range(N_CLASSES):
            for dd in np.linspace(0.4,1.6,61):
                w=bw.copy(); w[c]*=dd; s=ba(w)
                if s>best: best,bw=s,w
    bw=bw/bw.sum()*N_CLASSES
    if verbose: print("  decision weights:",np.round(bw,3),"| BA:",round(best,5))
    return bw,best


In [ ]:
# ============================================================
# 4. FT-Transformer with PLR numerical embeddings (pure PyTorch)
# ============================================================
class PLRTokenizer(nn.Module):
    """Numeric -> PLR: ReLU(Linear[sin(2pi v x), cos(2pi v x)]) ; categorical -> embedding + bias."""
    def __init__(self, n_num, cat_card, d, k, sigma):
        super().__init__()
        self.n_num=n_num; self.k=k
        self.freq=nn.Parameter(torch.randn(n_num,k)*sigma)              # learnable frequencies
        self.lin_w=nn.Parameter(torch.empty(n_num,2*k,d)); nn.init.normal_(self.lin_w,std=(2*k)**-0.5)
        self.lin_b=nn.Parameter(torch.zeros(n_num,d))
        self.n_cat=len(cat_card)
        if self.n_cat:
            offs=np.concatenate([[0],np.cumsum(cat_card)[:-1]]).astype(np.int64)
            self.register_buffer("cat_off",torch.tensor(offs))
            self.cat_emb=nn.Embedding(int(sum(cat_card)),d); nn.init.normal_(self.cat_emb.weight,std=d**-0.5)
            self.cat_b=nn.Parameter(torch.empty(self.n_cat,d)); nn.init.normal_(self.cat_b,std=d**-0.5)
    def forward(self, xn, xc):
        pre=2*math.pi*xn[:,:,None]*self.freq[None]                     # (B, n_num, k)
        per=torch.cat([torch.sin(pre),torch.cos(pre)],dim=-1)          # (B, n_num, 2k)
        num=torch.einsum("bnk,nkd->bnd", per, self.lin_w)+self.lin_b[None]
        toks=[F.relu(num)]                                            # (B, n_num, d)
        if self.n_cat:
            toks.append(self.cat_emb(xc+self.cat_off[None])+self.cat_b[None])
        return torch.cat(toks,dim=1)

class Block(nn.Module):
    def __init__(self,d,heads,ad,fd,rd,dff):
        super().__init__(); self.n1=nn.LayerNorm(d)
        self.attn=nn.MultiheadAttention(d,heads,dropout=ad,batch_first=True)
        self.n2=nn.LayerNorm(d); self.l1=nn.Linear(d,dff*2); self.l2=nn.Linear(dff,d)
        self.fd=nn.Dropout(fd); self.rd=nn.Dropout(rd)
    def forward(self,x):
        r=self.n1(x); a,_=self.attn(r,r,r,need_weights=False); x=x+self.rd(a)
        r=self.n2(x); u,v=self.l1(r).chunk(2,dim=-1); x=x+self.rd(self.l2(self.fd(u*F.relu(v))))  # ReGLU
        return x

class FTPLR(nn.Module):
    def __init__(self,n_num,cat_card,d=128,n_blocks=2,heads=8,ad=0.2,fd=0.1,rd=0.0,k=24,sigma=0.1,d_out=3):
        super().__init__()
        self.tok=PLRTokenizer(n_num,cat_card,d,k,sigma)
        self.cls=nn.Parameter(torch.empty(1,1,d)); nn.init.normal_(self.cls,std=d**-0.5)
        dff=int(d*4/3); self.blocks=nn.ModuleList([Block(d,heads,ad,fd,rd,dff) for _ in range(n_blocks)])
        self.hn=nn.LayerNorm(d); self.head=nn.Linear(d,d_out)
    def forward(self,xn,xc):
        x=self.tok(xn,xc); x=torch.cat([self.cls.expand(x.shape[0],-1,-1),x],dim=1)  # + CLS
        for b in self.blocks: x=b(x)
        return self.head(F.relu(self.hn(x[:,0])))
print("FT-PLR defined. Tokens =",len(NUM)+len(te_names),"num +",len(CAT),"cat + 1 CLS")


In [ ]:
# ============================================================
# 5. Per-fold preprocessing (leak-free TE + scaling) + training of one model
# ============================================================
from sklearn.preprocessing import TargetEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer

def build_fold_arrays(tr, va):
    enc=TargetEncoder(cv=5,smooth="auto",shuffle=True,random_state=SEED)   # refit on train only -> no leak
    Zt=enc.fit_transform(Xs_full.iloc[tr],y[tr]); Zv=enc.transform(Xs_full.iloc[va]); Ze=enc.transform(Xs_test)
    imp=SimpleImputer(strategy="median")
    Rt=imp.fit_transform(train.iloc[tr][NUM]); Rv=imp.transform(train.iloc[va][NUM]); Re=imp.transform(test[NUM])
    Nt=np.hstack([Rt,Zt]); Nv=np.hstack([Rv,Zv]); Ne=np.hstack([Re,Ze])   # 7 raw + 39 TE
    qt=QuantileTransformer(output_distribution="normal",n_quantiles=1000,subsample=200_000,random_state=SEED)
    Nt=qt.fit_transform(Nt).astype(np.float32); Nv=qt.transform(Nv).astype(np.float32); Ne=qt.transform(Ne).astype(np.float32)
    return Nt,Nv,Ne

def predict_proba(model,Xn,Xc,bs=8192):
    model.eval(); outs=[]
    with torch.no_grad():
        for i in range(0,len(Xn),bs):
            with torch.autocast("cuda",enabled=(CFG["use_amp"] and DEVICE=="cuda")):
                o=model(Xn[i:i+bs],Xc[i:i+bs])
            outs.append(F.softmax(o.float(),dim=1).cpu().numpy())
    return np.concatenate(outs,0)

def train_one(Nt,Xc_tr,y_tr,Nv,Xc_va,y_va,Ne,Xc_te,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xn_tr=torch.tensor(Nt,device=DEVICE); Xc_trg=torch.tensor(Xc_tr,device=DEVICE)
    Xn_va=torch.tensor(Nv,device=DEVICE); Xc_vag=torch.tensor(Xc_va,device=DEVICE)
    Xn_te=torch.tensor(Ne,device=DEVICE); Xc_teg=torch.tensor(Xc_te,device=DEVICE)
    yt=torch.tensor(y_tr,device=DEVICE,dtype=torch.long)
    cnt=np.bincount(y_tr,minlength=N_CLASSES)
    cw=torch.tensor(len(y_tr)/(N_CLASSES*cnt),device=DEVICE,dtype=torch.float32)   # balanced class weights
    model=FTPLR(Nt.shape[1],CAT_CARD,d=CFG["d_token"],n_blocks=CFG["n_blocks"],heads=CFG["n_heads"],
                ad=CFG["attn_dropout"],fd=CFG["ffn_dropout"],rd=CFG["res_dropout"],
                k=CFG["plr_k"],sigma=CFG["plr_sigma"]).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=CFG["weight_decay"])
    bs=CFG["batch_size"]; n=len(y_tr); spe=math.ceil(n/bs); total=spe*CFG["max_epochs"]
    warm=max(1,int(total*CFG["warmup_frac"]))
    def lr_at(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return 0.5*(1+math.cos(math.pi*p))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_at)
    crit=nn.CrossEntropyLoss(weight=cw)
    scaler=torch.cuda.amp.GradScaler(enabled=(CFG["use_amp"] and DEVICE=="cuda"))
    best_ba,best_state,bad=-1.0,None,0
    for ep in range(CFG["max_epochs"]):
        model.train(); perm=torch.randperm(n,device=DEVICE)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]; opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda",enabled=(CFG["use_amp"] and DEVICE=="cuda")):
                loss=crit(model(Xn_tr[idx],Xc_trg[idx]),yt[idx])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        ba=fast_balacc(y_va,predict_proba(model,Xn_va,Xc_vag).argmax(1))
        if ba>best_ba+1e-5: best_ba,best_state,bad=ba,copy.deepcopy(model.state_dict()),0
        else:
            bad+=1
            if bad>=CFG["patience"]: break
    model.load_state_dict(best_state)
    vpr=predict_proba(model,Xn_va,Xc_vag); tpr=predict_proba(model,Xn_te,Xc_teg)
    del model,Xn_tr,Xc_trg,Xn_va,Xc_vag,Xn_te,Xc_teg,yt
    if DEVICE=="cuda": torch.cuda.empty_cache()
    gc.collect()
    return vpr,tpr,best_ba,ep+1
print("fold functions ready.")


In [ ]:
# ============================================================
# 6. Full training: 5 folds x seed-averaging
# ============================================================
oof=np.zeros((len(train),N_CLASSES)); tp=np.zeros((len(test),N_CLASSES))
t0=time.time()
for f,(tr,va) in enumerate(FOLDS):
    tf=time.time(); Nt,Nv,Ne=build_fold_arrays(tr,va)
    oof_f=np.zeros((len(va),N_CLASSES)); tp_f=np.zeros((len(test),N_CLASSES))
    for s in CFG["seed_avg"]:
        vpr,tpr,ba,ep=train_one(Nt,XC_full[tr],y[tr],Nv,XC_full[va],y[va],Ne,XC_test,seed=s)
        oof_f+=vpr/len(CFG["seed_avg"]); tp_f+=tpr/len(CFG["seed_avg"])
        print(f"    fold {f+1} seed {s}: val BA={ba:.5f} (epochs={ep})")
    oof[va]=oof_f; tp+=tp_f/CFG["n_folds"]
    print(f"  fold {f+1}: BA(argmax)={fast_balacc(y[va],oof_f.argmax(1)):.5f}  ({(time.time()-tf)/60:.1f} min)")
    del Nt,Nv,Ne; gc.collect()
print(f"\n[FT-PLR] OOF BA (argmax) = {fast_balacc(y,oof.argmax(1)):.5f}   | total {(time.time()-t0)/60:.1f} min")
np.savez_compressed(OUT/"oof_test_ftplr.npz", y=y, oof_ftplr=oof, test_ftplr=tp)
bias_report(y, oof.argmax(1), "FT-PLR OOF")
gc.collect()


In [ ]:
# ============================================================
# 7. Submission (argmax: on a class-weighted model the decision rule adds nothing)
# ============================================================
test_pred=tp.argmax(1)
sub=pd.DataFrame({ID:test[ID],TARGET:[I2C[i] for i in test_pred]})
assert list(sub.columns)==[ID,TARGET] and len(sub)==len(test) and set(sub[TARGET])<=set(CLASSES)
SUB_PATH=Path("/kaggle/working/submission.csv")
sub.to_csv(SUB_PATH,index=False)
print("Wrote:",SUB_PATH)
print("OOF BA (argmax) =",round(fast_balacc(y,oof.argmax(1)),5))
print("predicted distribution:",sub[TARGET].value_counts(normalize=True).round(4).to_dict())
print("train distribution    :",dict(zip(CLASSES,priors.round(4))))
